```
<08-4_data_renewal.ipynb>

제미나이 의존도: 20-30%

LR Scheduler(CosineAnnealingLR) StratifiedKFold에 이어서 강력한 데이터 증강을 추가해봤다.
RandomBrightnessContrast, GaussNoise, ColorJitter

하지만 결과는 증강하기 전보다 좋지 않았다.
이 증강은 이미지의 색감과 화질을 왜곡하다보니 모델 입장에서는 오히려 학습이 방해된 것이다.
많은 증강기법을 추가하는 게 정답이 아니라는 것을 깨달았다.

다음에는 조기 종료와 가중치 동결 및 해제를 해봐야겠다.
```

In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold

N_SPLITS = 5
BATCH_SIZE = 64
EPOCHS = 3

path = "images/kaggle_data/dataset/"

fruits = ["apple", "banana", "orange"]

classes = []

data = []

for fruit in fruits:
    path = f"images/kaggle_data/dataset/{fruit}"
    categories = [f"fresh_{fruit}", f"rotten_{fruit}"]
    classes.extend(categories)
    for category in categories:
        folder_path = os.path.join(path, category)
        for image in os.listdir(folder_path):
            if image.endswith(("jpg", "jpeg", "png")):
                data.append({
                    "image_path": os.path.join(folder_path, image),
                    "label_name": category
                })

df = pd.DataFrame(data)

label_map = {name: i for i, name in enumerate(classes)}

df['label'] = df['label_name'].map(label_map)
print(classes)
print(len(df))

train_val_df, test_df = train_test_split(
    df, test_size=0.1, stratify=df['label'], random_state=42, shuffle=True
)

train_val_df = train_val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

skf = StratifiedKFold(n_splits=N_SPLITS, random_state=42, shuffle=True)

df['fold'] = -1

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(train_val_df, train_val_df['label'])):
    train_val_df.loc[val_idx, 'fold'] = fold_idx

['fresh_apple', 'rotten_apple', 'fresh_banana', 'rotten_banana', 'fresh_orange', 'rotten_orange']
16831


In [2]:
from torch.utils.data import Dataset, DataLoader
import cv2

class FruitsDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = row['image_path']
        image = cv2.imread(image)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            aug = self.transform(image=image)
            image = aug['image']

        label = int(row['label'])
        return image, torch.tensor(label, dtype=torch.long)

In [12]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

train_transform = A.Compose([
    A.Resize(128, 128), 
    A.HorizontalFlip(p=0.5), 
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5), 
    A.GaussNoise(std_range=(0.02, 0.1), p=0.5), 
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5), 
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), 
    ToTensorV2()
])

val_test_transform = A.Compose([
    A.Resize(128, 128), 
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), 
    ToTensorV2()
])

In [13]:
test_dataset = FruitsDataset(df=test_df, transform=val_test_transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [7]:
import torch

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"device: {device}")

device: mps


In [14]:
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

for fold in range(N_SPLITS):
    print(f"===Fold {fold} START===")
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, len(classes))
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    scheduler = CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-6)

    train_df = train_val_df[train_val_df['fold'] != fold].reset_index(drop=True)
    val_df = train_val_df[train_val_df['fold'] == fold].reset_index(drop=True)
    
    train_dataset = FruitsDataset(df=train_df, transform=train_transform)
    val_dataset = FruitsDataset(df=val_df, transform=val_test_transform)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    best_val_loss = float('inf')

    for epoch in range(EPOCHS):
        # ---Train Start---
        model.train()
        train_loss = 0.0
        train_correct = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            preds = outputs.argmax(1)
            train_correct += (preds == labels).sum().item()
            train_loss += loss.item() * images.size(0)

            scheduler.step()

            current_lr = scheduler.get_last_lr()[0]

        print(f"Epoch {epoch+1}/{EPOCHS} [Train] Loss: {train_loss/len(train_dataset):.4f}, "\
              f"Acc: {train_correct/len(train_dataset) * 100:.2f}%, lr: {current_lr}")
        # ---Train End---

        # ---Validation Start---
        model.eval()
        val_loss = 0.0
        val_correct = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                
                outputs = model(images)
                loss = criterion(outputs, labels)

                preds = outputs.argmax(1)
                val_correct += (preds == labels).sum().item()
                val_loss += loss.item() * images.size(0)

        val_loss /= len(val_dataset)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), f'best_model_kfold{fold}.pth')

        print(f"[Validation] Loss: {val_loss:.4f}, Acc: {val_correct/len(val_dataset) * 100:.2f}%")
        # ---Validation End---
        
    # ---Test Start---
    best_model = models.resnet18(weights=None)
    best_model.fc = nn.Linear(best_model.fc.in_features, len(classes))
    best_model.load_state_dict(torch.load(f'best_model_kfold{fold}.pth'))
    best_model.to(device)
    best_model.eval()

    test_correct = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = best_model(images)

            preds = outputs.argmax(1)
            test_correct += (preds == labels).sum().item()

    print(f"[Test] Acc: {test_correct/len(test_dataset) * 100:.2f}%")
    
    # ---Test End---
        
        

===Fold 0 START===
Epoch 1/3 [Train] Loss: 0.2232, Acc: 92.42%, lr: 0.0009046039886902865
[Validation] Loss: 0.1444, Acc: 95.12%
Epoch 2/3 [Train] Loss: 0.1472, Acc: 94.93%, lr: 0.0006548539886903229
[Validation] Loss: 0.0320, Acc: 98.88%
Epoch 3/3 [Train] Loss: 0.0975, Acc: 96.80%, lr: 0.0003461460113097342
[Validation] Loss: 0.0182, Acc: 99.54%
[Test] Acc: 99.47%
===Fold 1 START===
Epoch 1/3 [Train] Loss: 0.2525, Acc: 91.36%, lr: 0.0009046039886902865
[Validation] Loss: 0.1362, Acc: 95.51%
Epoch 2/3 [Train] Loss: 0.1385, Acc: 95.19%, lr: 0.0006548539886903229
[Validation] Loss: 0.0465, Acc: 98.61%
Epoch 3/3 [Train] Loss: 0.1148, Acc: 96.12%, lr: 0.0003461460113097342
[Validation] Loss: 0.0218, Acc: 99.41%
[Test] Acc: 99.35%
===Fold 2 START===
Epoch 1/3 [Train] Loss: 0.2469, Acc: 91.59%, lr: 0.0009046039886902865
[Validation] Loss: 0.1347, Acc: 94.55%
Epoch 2/3 [Train] Loss: 0.1482, Acc: 95.00%, lr: 0.0006548539886903229
[Validation] Loss: 0.0576, Acc: 97.92%
Epoch 3/3 [Train] Loss: 0